# CITYNEXUS — Training Notebook (Colab)

End-to-end training pipeline for the CITYNEXUS multi-agent urban simulation:

1. **Setup** — install deps + bring in the `citynexus` package
2. **Quick env demo** — run a few ticks to see what's in the box
3. **Train** — run the `TrainingPipeline` with adaptive curriculum + persistent memory + verifier-gated rewards
4. **Plots** — reward curves, city score, difficulty trajectory, per-agent breakdown
5. **Evaluate** — baseline (no memory, fixed difficulty) vs trained (memory + curriculum)
6. **Optional** — Unsloth + TRL scaffolding for fine-tuning a small LLM as the policy

Each cell is self-contained: paste them one at a time into Colab, or upload this notebook directly.

## 1. Setup

**Option A (recommended)** — clone your CITYNEXUS repo from GitHub:

In [ ]:
# Replace with your fork's URL after pushing the project to GitHub.
GITHUB_URL = "https://github.com/YOUR_USERNAME/CITYNEXUS.git"

import os
if not os.path.exists("CITYNEXUS"):
    !git clone {GITHUB_URL}
%cd CITYNEXUS

**Option B** — upload `citynexus.zip` (zip the project folder locally first):

In [ ]:
# from google.colab import files
# uploaded = files.upload()                       # pick citynexus.zip
# !unzip -q citynexus.zip -d /content/
# %cd /content/CITYNEXUS

In [ ]:
# Install deps + the package itself.
!pip install -q numpy matplotlib pandas
!pip install -q -e .

In [ ]:
# Sanity check: top-level imports work.
from citynexus import (
    CityNexusEnv, EnvConfig,
    MultiAgentCoordinator, CoordinatorConfig,
    DeliveryAgent, TrafficAgent, EmergencyAgent, PoliceAgent, PlannerAgent,
    Verifier, VerificationContext,
    MultiAgentRewardSystem, RewardSystemConfig,
    MemoryStore, MemoryWriter,
    AdversarialGenerator, Curriculum, EpisodeRunner,
    TrainingPipeline, TrainingConfig, EvalConfig, Evaluator,
    PolicyBundle, CallablePolicy,
    AgentRole,
)
print("citynexus imports OK")

## 2. Quick env demo

30 ticks of the simulation with the heuristic agents — confirms everything is wired.

In [ ]:
env = CityNexusEnv(EnvConfig(width=20, height=20, seed=11, max_ticks=100))
agents = [DeliveryAgent(), TrafficAgent(), EmergencyAgent(), PoliceAgent(), PlannerAgent()]
coord = MultiAgentCoordinator(env, agents, CoordinatorConfig(
    seed=11, delivery_spawn_rate=0.30, incident_spawn_rate=0.10,
))
coord.reset()
for _ in range(30):
    coord.step()

ctx = coord.ctx
print(f"After 30 ticks:")
print(f"  weather:           {env.state.weather.value}")
print(f"  active accidents:  {len(env.state.accidents)}")
print(f"  active incidents:  {len(ctx.incidents)}")
print(f"  deliveries:        {sum(1 for d in ctx.deliveries.values() if d.status.value=='delivered')} done"
      f" / {sum(1 for d in ctx.deliveries.values() if d.status.value=='failed')} failed"
      f" / {sum(1 for d in ctx.deliveries.values() if d.is_open)} open")
print(f"  messages routed:   {ctx.bus.stats.get('sent', 0)}")
print(f"  per-role units:    "
      f"{sum(1 for u in ctx.units.values() if u.kind.value=='ambulance')} ambulances, "
      f"{sum(1 for u in ctx.units.values() if u.kind.value=='police_car')} police, "
      f"{sum(1 for u in ctx.units.values() if u.kind.value=='delivery_van')} vans")

## 3. Train

Runs N episodes with:
- adaptive curriculum (difficulty drifts toward agent's competence)
- persistent memory (high-risk zones + past failures saved to disk, agents query in `decide()`)
- 3-layer verifier (programmatic / system-state / semantic) gating per-tick rewards
- multi-agent reward decomposition (per-role + global city score + process-aware)

Each episode logs to `runs/training.jsonl` (per-episode metrics) — we'll plot from `pipeline.logger`.

In [ ]:
import os
os.makedirs("runs", exist_ok=True)

def make_agents():
    return [DeliveryAgent(), TrafficAgent(), EmergencyAgent(), PoliceAgent(), PlannerAgent()]

config = TrainingConfig(
    n_episodes=40,
    max_ticks_per_episode=80,
    curriculum_target=0.55,
    starting_difficulty=0.20,
    use_memory=True,
    memory_path="runs/memory.json",
    log_dir="runs",
    log_window=5,
    seed=42,
    console=True,
)
pipeline = TrainingPipeline(agents_factory=make_agents, config=config)
summary = pipeline.train()

print()
print(f"Training summary:")
print(f"  episodes:                   {summary.n_episodes}")
print(f"  final difficulty:           {summary.final_difficulty:.2f}")
print(f"  mean score (all):           {summary.mean_score:.3f}")
print(f"  last-5-episode avg score:   {summary.last_window_avg_score:.3f}")
print(f"  mean delivery success rate: {summary.mean_delivery_success:.3f}")
print(f"  cumulative reward by role:")
for role, val in summary.cumulative_reward_per_role.items():
    print(f'    {role:<10s} {val:+8.1f}')

## 4. Plots — reward curves + city score + difficulty

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    'figure.facecolor': '#0d0d1d',
    'axes.facecolor':   '#11111f',
    'axes.edgecolor':   '#22223a',
    'axes.labelcolor':  '#e6e6f0',
    'text.color':       '#e6e6f0',
    'xtick.color':      '#b6b6c8',
    'ytick.color':      '#b6b6c8',
    'grid.color':       '#22223a',
    'axes.grid':        True,
    'font.family':      'sans-serif',
    'font.size':        10,
})

records = pipeline.logger.all()
ep        = [r['episode'] for r in records]
score     = [r['score'] for r in records]
succ      = [r['delivery_success_rate'] for r in records]
diff      = [r['difficulty'] for r in records]
summed_r  = [r['summed_reward'] for r in records]
city_avg  = [r['avg_city_score'] for r in records]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('CITYNEXUS Training', color='#21d4fd', fontsize=14, fontweight='bold')

axes[0,0].plot(ep, score, color='#7c5cff', linewidth=2, label='per-episode')
if len(score) >= 5:
    rolling = np.convolve(score, np.ones(5)/5, mode='valid')
    axes[0,0].plot(ep[4:], rolling, color='#21d4fd', linewidth=2.5, label='rolling-5 avg')
axes[0,0].axhline(0.55, color='#ffb020', linestyle='--', alpha=0.6, label='curriculum target')
axes[0,0].set_title('Episode score')
axes[0,0].set_xlabel('episode'); axes[0,0].set_ylabel('overall score [0..1]')
axes[0,0].legend(facecolor='#16162a', edgecolor='#22223a')

axes[0,1].plot(ep, diff, color='#ff3b6b', linewidth=2)
axes[0,1].fill_between(ep, 0, diff, color='#ff3b6b', alpha=0.15)
axes[0,1].set_title('Curriculum difficulty')
axes[0,1].set_xlabel('episode'); axes[0,1].set_ylabel('difficulty [0..1]')

axes[1,0].plot(ep, summed_r, color='#10b981', linewidth=2)
axes[1,0].set_title('Summed reward (all roles)')
axes[1,0].set_xlabel('episode'); axes[1,0].set_ylabel('cumulative reward')

axes[1,1].plot(ep, succ, color='#f59e0b', linewidth=2, label='success rate')
axes[1,1].plot(ep, city_avg, color='#a78bfa', linewidth=2, label='avg city score')
axes[1,1].set_title('Outcomes')
axes[1,1].set_xlabel('episode'); axes[1,1].set_ylabel('rate / score')
axes[1,1].legend(facecolor='#16162a', edgecolor='#22223a')

plt.tight_layout()
plt.savefig('runs/training_curves.png', dpi=140, facecolor='#0d0d1d', bbox_inches='tight')
plt.show()
print('saved → runs/training_curves.png')

In [ ]:
# Per-agent reward decomposition over episodes.
import collections
per_role = collections.defaultdict(list)
for r in records:
    cum = r.get('cumulative_per_agent', {})
    for role in ['delivery','traffic','emergency','police','planner']:
        per_role[role].append(cum.get(role, 0.0))

fig, ax = plt.subplots(figsize=(13, 4))
fig.suptitle('Per-agent cumulative reward (per episode)', color='#21d4fd', fontsize=12, fontweight='bold')
colors = {'delivery':'#f59e0b','traffic':'#3b82f6','emergency':'#10b981','police':'#a78bfa','planner':'#21d4fd'}
for role, vals in per_role.items():
    ax.plot(ep, vals, label=role, color=colors[role], linewidth=2)
ax.set_xlabel('episode'); ax.set_ylabel('reward summed within episode')
ax.legend(facecolor='#16162a', edgecolor='#22223a', ncol=5)
plt.tight_layout()
plt.savefig('runs/per_agent_reward.png', dpi=140, facecolor='#0d0d1d', bbox_inches='tight')
plt.show()
print('saved → runs/per_agent_reward.png')

## 5. Evaluate — baseline vs trained

Same fixed difficulty + same seeds for both arms — only the agent setup differs.

- **baseline** = fresh agents, no memory file (the heuristic with no learned hotspots)
- **trained**  = agents that get to read the memory built up during training above

In [ ]:
# Eval config — fixed difficulty so both arms see the same scenarios.
eval_cfg = EvalConfig(
    n_episodes=12,
    max_ticks_per_episode=80,
    base_seed=2000,
    fixed_difficulty=0.45,
)

# Baseline arm — no persistent memory.
def make_agents_baseline():
    return [DeliveryAgent(), TrafficAgent(), EmergencyAgent(), PoliceAgent(), PlannerAgent()]

evaluator = Evaluator(agents_factory=make_agents_baseline, config=eval_cfg)
baseline = evaluator.evaluate(name='baseline')
print(f'baseline:  mean_score={baseline.mean_score:.3f} (±{baseline.std_score:.3f})  '
      f'success_rate={baseline.mean_success_rate:.3f}')

In [ ]:
# Trained arm — same agent class but with the warm memory store loaded.
# (We attach the warm memory by patching the Coordinator inside the runner;
#  simplest is to use a dedicated Evaluator with a custom env_factory + agent
#  factory that has a coordinator hook. For demo, we reuse Evaluator and just
#  show the loaded memory has zones for the agents to react to.)
warm_memory = MemoryStore(path='runs/memory.json')
print(f'warm memory loaded: {len(warm_memory)} records')
print(f'  by kind: {warm_memory.stats()["by_kind"]}')
print(f'  hottest zones (top 5):')
for z in warm_memory.hottest_zones(5):
    print(f'    risk={z.risk_score:.2f}  cell={z.coords[0] if z.coords else "?"}'
          f'  factors={z.risk_factors[:2]}  samples={z.sample_count}')

In [ ]:
# Trained eval: build a custom evaluator that wires warm_memory into each coord.
from citynexus.scenarios.runner import EpisodeRunner, EpisodeRunnerConfig

scores_trained, success_trained = [], []
gen = AdversarialGenerator(grid_size=(20, 20), episode_length=80, seed=eval_cfg.base_seed)
for i in range(eval_cfg.n_episodes):
    seed = eval_cfg.base_seed + i
    env = CityNexusEnv(EnvConfig(width=20, height=20, seed=seed, max_ticks=80))
    agents = make_agents_baseline()
    coord = MultiAgentCoordinator(
        env, agents,
        CoordinatorConfig(seed=seed, delivery_spawn_rate=0.30, incident_spawn_rate=0.10),
        memory=warm_memory,                       # ← key difference
    )
    scenario = gen.generate(eval_cfg.fixed_difficulty, seed=seed)
    metrics = EpisodeRunner(coord, scenario, EpisodeRunnerConfig(max_ticks=80)).run()
    scores_trained.append(metrics.overall_score)
    success_trained.append(metrics.delivery_success_rate)

import statistics
print(f'trained:   mean_score={statistics.mean(scores_trained):.3f}'
      f' (±{statistics.stdev(scores_trained):.3f})  '
      f'success_rate={statistics.mean(success_trained):.3f}')

In [ ]:
# Side-by-side bar chart.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle('Baseline vs Memory-Trained (same seeds, fixed difficulty)',
             color='#21d4fd', fontsize=13, fontweight='bold')

labels  = ['baseline', 'trained']
scores  = [baseline.mean_score, statistics.mean(scores_trained)]
errs    = [baseline.std_score, statistics.stdev(scores_trained)]
succs   = [baseline.mean_success_rate, statistics.mean(success_trained)]

bars = axes[0].bar(labels, scores, yerr=errs, color=['#7a7a92','#21d4fd'],
                   edgecolor='#fff', capsize=6)
axes[0].set_title('mean overall score')
axes[0].set_ylim(0, 1)
for b, v in zip(bars, scores):
    axes[0].text(b.get_x() + b.get_width()/2, v + 0.02, f'{v:.3f}', ha='center')

bars = axes[1].bar(labels, succs, color=['#7a7a92','#10b981'], edgecolor='#fff')
axes[1].set_title('mean delivery success rate')
axes[1].set_ylim(0, 1)
for b, v in zip(bars, succs):
    axes[1].text(b.get_x() + b.get_width()/2, v + 0.02, f'{v:.3f}', ha='center')

plt.tight_layout()
plt.savefig('runs/baseline_vs_trained.png', dpi=140, facecolor='#0d0d1d', bbox_inches='tight')
plt.show()
print('saved → runs/baseline_vs_trained.png')

## 6. Optional — Unsloth + TRL scaffolding

All cells above run on CPU. To wire in an **actual LLM policy** trained with Unsloth + HF TRL:

1. Switch Colab runtime → **GPU (T4 / A100)**
2. Install Unsloth + TRL
3. Pick a small base model (Qwen2.5-0.5B or Qwen2.5-1.5B fits on T4)
4. Convert the trajectories collected above into a TRL-compatible dataset
5. Fine-tune with `SFTTrainer` (or `GRPOTrainer` for online RL)
6. Wrap the trained model in a `CallablePolicy` and re-run `Evaluator.compare`

The block below is the integration scaffold — uncomment and adapt to actually run.

In [ ]:
# ---- INSTALL (uncomment after switching to GPU runtime) ----
# !pip install -q --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# !pip install -q trl bitsandbytes accelerate xformers

In [ ]:
# ---- CONVERT TRAJECTORIES → SFT DATASET ----
# Filter to high-reward (obs, action) pairs from the training trajectories.
# These become positive demonstrations for supervised fine-tuning.
#
# def obs_to_prompt(role: str, obs: dict) -> str:
#     """Render an agent observation as a text prompt the LLM can read."""
#     # Example: role + key numeric features + visible entities
#     parts = [f"You are the {role} agent. Current state:"]
#     parts.append(f"  weather={obs.get('weather','?')}, hour={obs.get('hour_of_day','?')}")
#     parts.append(f"  visible accidents={obs.get('n_visible_accidents', obs.get('n_accidents', 0))}")
#     parts.append(f"  congestion={obs.get('congestion_ratio', [0])[0]:.2f}")
#     parts.append("\nWhat action do you take? Respond with one of: ...")
#     return "\n".join(parts)
#
# def action_to_completion(actions) -> str:
#     return ", ".join(type(a).__name__ for a in actions if type(a).__name__ != 'NoOp') or 'NoOp'
#
# samples = []
# for traj in pipeline.trajectories:
#     for role, transitions in traj.role_transitions.items():
#         for tr in transitions:
#             if tr.reward >= 0.20 and not tr.info.get('gated'):
#                 samples.append({
#                     'prompt':     obs_to_prompt(role, tr.obs),
#                     'completion': action_to_completion(tr.actions),
#                 })
# print(f'Filtered {len(samples)} positive (obs, action) demonstrations.')

In [ ]:
# ---- LOAD UNSLOTH MODEL + LORA ADAPTER ----
# from unsloth import FastLanguageModel
# import torch
#
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name='unsloth/Qwen2.5-0.5B-Instruct',
#     max_seq_length=1024,
#     dtype=None,
#     load_in_4bit=True,
# )
# model = FastLanguageModel.get_peft_model(
#     model,
#     r=16, lora_alpha=16, lora_dropout=0,
#     target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
#     use_gradient_checkpointing='unsloth',
# )

In [ ]:
# ---- TRAIN WITH TRL SFTTrainer ----
# from datasets import Dataset
# from trl import SFTTrainer, SFTConfig
#
# ds = Dataset.from_list([{'text': s['prompt'] + '\n' + s['completion']} for s in samples])
# trainer = SFTTrainer(
#     model=model, tokenizer=tokenizer, train_dataset=ds,
#     args=SFTConfig(
#         output_dir='runs/sft',
#         per_device_train_batch_size=2, gradient_accumulation_steps=4,
#         num_train_epochs=2, learning_rate=2e-4, logging_steps=10,
#         max_seq_length=1024, dataset_text_field='text',
#         fp16=not torch.cuda.is_bf16_supported(),
#         bf16=torch.cuda.is_bf16_supported(),
#     ),
# )
# trainer_stats = trainer.train()
# print('SFT done.')

In [ ]:
# ---- WRAP TRAINED MODEL AS A POLICY + EVAL ----
# def llm_inference(role: str, obs: dict, ctx) -> list:
#     prompt = obs_to_prompt(role, obs)
#     inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
#     out = model.generate(**inputs, max_new_tokens=20, do_sample=False)
#     text = tokenizer.decode(out[0], skip_special_tokens=True)
#     # Parse `text` → typed Action list.  Fall back to NoOp on parse failure.
#     from citynexus.agents.base import NoOp
#     return [NoOp()]   # ← replace with real parser
#
# trained_bundle = PolicyBundle({
#     'delivery':  CallablePolicy('delivery',  lambda obs, ctx: llm_inference('delivery', obs, ctx)),
#     'emergency': CallablePolicy('emergency', lambda obs, ctx: llm_inference('emergency', obs, ctx)),
# })
# cmp = evaluator.compare(trained_bundle=trained_bundle)
# print(cmp.summary())

## Done

Plots are saved to `runs/`:
- `training_curves.png` — score, difficulty, summed reward, success-rate over episodes
- `per_agent_reward.png` — per-role cumulative reward each episode
- `baseline_vs_trained.png` — held-out comparison

Memory snapshot is saved to `runs/memory.json` — the agents' learned hotspots persist across runs.

To export everything for the hackathon submission, zip and download `runs/`:

```python
!zip -q -r runs.zip runs
from google.colab import files
files.download('runs.zip')
```